In [ ]:
%%writefile build_known_subset.py
"""
Build the "known" subset for ConflictBank Experiment 3.3 replica.

Filters questions from Warrieryes/CB_qa down to those that BOTH non-finetuned
Qwen2.5-0.5B and Qwen2.5-3B answer correctly under BOTH evaluation conditions:
  (a) no evidence prompt
  (b) with default_evidence prepended

Output: a JSON file of up to N_TARGET (default 100) qualifying rows, with
their indices into the original dataset, plus the prebuilt prompts and gold
labels — ready to feed into the four finetuned models for Figure 6 scoring.
"""

import os
import json
import gc
import argparse
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm


# ----------------------------- Config -----------------------------
DATASET_ID   = "Warrieryes/CB_qa"
BASE_MODELS  = ["Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-3B"]
N_TARGET     = 100      # known-subset size to aim for
CHUNK_SIZE   = 512      # candidate rows scored per pass
BATCH_SIZE   = 16       # forward-pass batch size for the model
MAX_LEN      = 2048     # truncation cap for prompts
RANDOM_SEED  = 42
OUT_PATH     = "./known_subset.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32


# ----------------------------- Prompts ----------------------------
PROMPT_NO_EVIDENCE = (
    "According to your knowledge, choose the best choice from the following options."
)
PROMPT_WITH_EVIDENCE = (
    "According to the evidence provided and your knowledge, "
    "choose the best choice from the following options."
)

def build_prompt(row, evidence_field=None):
    q = row["question"]
    A, B, C, D = row["options"]
    if evidence_field is None:
        return (f"{PROMPT_NO_EVIDENCE}\n\n"
                f"Question: {q}\nA. {A}\nB. {B}\nC. {C}\nD. {D}\nAnswer:")
    ev = row[evidence_field]
    return (f"{PROMPT_WITH_EVIDENCE}\n\n"
            f"Evidence: {ev}\n"
            f"Question: {q}\nA. {A}\nB. {B}\nC. {C}\nD. {D}\nAnswer:")


# ----------------------------- Scoring ----------------------------
@torch.no_grad()
def predict_choices(model, tokenizer, prompts, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    """Return list of predicted letters ('A'/'B'/'C'/'D') for each prompt."""
    # Resolve token id for each option (leading space matters for Qwen BPE)
    label_ids = []
    for label in [" A", " B", " C", " D"]:
        ids = tokenizer.encode(label, add_special_tokens=False)
        label_ids.append(ids[-1])

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # last position == answer slot

    preds = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(model.device)
        logits = model(**enc).logits[:, -1, :]
        choice_logits = logits[:, label_ids]
        choice_idx    = choice_logits.argmax(dim=-1).tolist()
        preds.extend(["ABCD"[k] for k in choice_idx])
    return preds


def score_model_on_chunk(repo, chunk_rows):
    """Load a base model, score all chunk_rows under both conditions,
    return (preds_no_ev, preds_with_ev) lists. Frees model on exit."""
    print(f"\n  Loading {repo}...")
    tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        repo, torch_dtype=dtype, trust_remote_code=True
    ).to(device).eval()

    prompts_no_ev   = [build_prompt(r, None)               for r in chunk_rows]
    prompts_with_ev = [build_prompt(r, "default_evidence") for r in chunk_rows]

    preds_no_ev   = predict_choices(mdl, tok, prompts_no_ev)
    preds_with_ev = predict_choices(mdl, tok, prompts_with_ev)

    del mdl, tok
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return preds_no_ev, preds_with_ev


# ----------------------------- Main loop --------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_target",  type=int, default=N_TARGET)
    parser.add_argument("--chunk",     type=int, default=CHUNK_SIZE)
    parser.add_argument("--batch",     type=int, default=BATCH_SIZE)
    parser.add_argument("--seed",      type=int, default=RANDOM_SEED)
    parser.add_argument("--out",       type=str, default=OUT_PATH)
    args = parser.parse_args()

    print(f"Device: {device} | dtype: {dtype}")
    print(f"Target known-subset size: {args.n_target}")

    # Load dataset and create a deterministic shuffled index list
    print(f"\nLoading {DATASET_ID}...")
    ds = load_dataset(DATASET_ID, split="train")
    print(f"  Total rows: {len(ds):,}")
    print(f"  Columns:    {ds.column_names}")

    rng = np.random.default_rng(args.seed)
    shuffled = rng.permutation(len(ds)).tolist()

    # Iterate in chunks: score → filter → accumulate
    accepted = []   # list of dicts: {"orig_index", row fields..., gold}
    cursor   = 0
    pbar     = tqdm(total=args.n_target, desc="Building known subset")

    while len(accepted) < args.n_target and cursor < len(shuffled):
        chunk_indices = shuffled[cursor: cursor + args.chunk]
        chunk_rows    = [ds[int(i)] for i in chunk_indices]
        cursor += args.chunk
        gold = [r["correct_option"] for r in chunk_rows]

        # Score both base models on this chunk
        per_model_correct = []  # list of bool-arrays, one per base model
        for repo in BASE_MODELS:
            preds_no, preds_yes = score_model_on_chunk(repo, chunk_rows)
            correct_mask = [
                (preds_no[k] == gold[k]) and (preds_yes[k] == gold[k])
                for k in range(len(chunk_rows))
            ]
            per_model_correct.append(correct_mask)

        # Intersection: row qualifies iff every base model got it right both ways
        for k in range(len(chunk_rows)):
            if all(m[k] for m in per_model_correct):
                row = chunk_rows[k]
                accepted.append({
                    "orig_index":      int(chunk_indices[k]),
                    "question":        row["question"],
                    "options":         row["options"],
                    "correct_option":  row["correct_option"],
                    "replace_option":  row.get("replace_option"),
                    "uncertain_option": row.get("uncertain_option"),
                    "default_evidence": row["default_evidence"],
                    # Conflict-evidence fields kept for downstream experiments
                    "misinformation_conflict_evidence":
                        row.get("misinformation_conflict_evidence_evidence")
                        or row.get("misinformation_conflict_evidence"),
                    "temporal_conflict_evidence":
                        row.get("temporal_conflict_evidence"),
                    "semantic_conflict_evidence":
                        row.get("semantic_conflict_evidence"),
                })
                pbar.update(1)
                if len(accepted) >= args.n_target:
                    break

        # Progress hint
        rate = len(accepted) / max(1, cursor)
        tqdm.write(f"  [{cursor:,} scanned] kept {len(accepted)} "
                   f"(yield {rate*100:.1f}%)")

    pbar.close()

    # Save
    out = {
        "n_target":   args.n_target,
        "n_actual":   len(accepted),
        "n_scanned":  cursor,
        "seed":       args.seed,
        "base_models": BASE_MODELS,
        "rows":       accepted,
    }
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    with open(args.out, "w") as f:
        json.dump(out, f, indent=2)

    print(f"\nDone. Kept {len(accepted)}/{args.n_target} after scanning {cursor:,} rows.")
    print(f"Saved to {args.out}")
    if len(accepted) < args.n_target:
        print("WARNING: did not reach target. Increase chunk size or accept smaller subset.")


if __name__ == "__main__":
    main()

Writing build_known_subset.py


In [ ]:
!python build_known_subset.py --n_target 100 --out known_subset.json

Device: cuda | dtype: torch.float16
Target known-subset size: 100

Loading Warrieryes/CB_qa...
QA_dataset.json: 100% 5.79G/5.79G [00:24<00:00, 236MB/s]
Generating train split: 553117 examples [00:14, 37224.43 examples/s]
  Total rows: 553,117
  Columns:    ['relation', 'subject', 'subject_description', 'semantic_description', 'object', 'object_description', 'replaced_object', 'replaced_description', 'default_claim', 'default_evidence_category', 'default_evidence', 'misinformation_conflict_claim', 'misinformation_conflict_evidence_category', 'misinformation_conflict_evidence_evidence', 'temporal_conflict_time_span', 'temporal_conflict_claim', 'temporal_conflict_evidence_category', 'temporal_conflict_evidence', 'semantic_conflict_claim', 'semantic_conflict_evidence_category', 'semantic_conflict_evidence', 'question', 'options', 'correct_option', 'replace_option', 'uncertain_option']
Building known subset:   0% 0/100 [00:00<?, ?it/s]
  Loading Qwen/Qwen2.5-0.5B...

config.json: 100% 681/6